# 99.3 Reactions: Exercises

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Reaction calls, parameter changes and simulation comparisons.

**Practises:** reaction networks, intermediates and reversible reactions in a `Cstr`.

1. An intermediate reaction product
2. Reverse reactions (optional)

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

```{note}
**Optional material.** Chapter 99 is not part of the main sequence. Work through it after the related main topic or whenever time permits.
```

## Starter tank

The supplied closed CSTR holds 1 mL, initially `[A, B, C] = [1, 0, 0]` mol/m³. It has no flow or binding and runs for 100 s.

```{tip}
Run the cell below once with **Shift+Enter** to set up the supplied tank model for this exercise.
```

In [ ]:
%matplotlib inline
from CADETProcess.processModel import ComponentSystem, Cstr, FlowSheet, Process, MassActionLaw
from CADETProcess.simulator import Cadet
import copy
import numpy as np

component_system = ComponentSystem(["A", "B", "C"])
reactor = Cstr(component_system, "reactor")
reactor.init_liquid_volume = 1e-6
reactor.const_solid_volume = 0
reactor.c = [1, 0, 0]
flow_sheet = FlowSheet(component_system)
flow_sheet.add_unit(reactor)
checkpoint = Process(flow_sheet, "batch_checkpoint")
checkpoint.cycle_time = 100

## Exercise 1: an intermediate

Use $A\xrightarrow{0.08\,s^{-1}}B\xrightarrow{0.06\,s^{-1}}C$.

Predict the concentration curves. Add both reactions to a [MassActionLaw](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.MassActionLaw.html) using [`add_reaction()`](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.MassActionLaw.add_reaction.html) with component names such as `components=["A", "B"]`, attach it as `reactor.bulk_reaction_model`, then simulate. Does $c_A+c_B+c_C$ stay constant?

In [ ]:
process = copy.deepcopy(checkpoint)
reactions = MassActionLaw(process.component_system)
reactions.add_reaction(components=["A", "B"], coefficients=[-1, 1], k_fwd=0.08, k_bwd=0)
reactions.add_reaction(components=["B", "C"], coefficients=[-1, 1], k_fwd=0.06, k_bwd=0)
process.flow_sheet.reactor.bulk_reaction_model = reactions
result = Cadet().simulate(process)
solution = result.solution.reactor.outlet
_ = solution.plot()
print("Largest conservation error:", np.max(np.abs(solution.solution.sum(axis=1) - 1)))

```{dropdown} Answer
A decays; B first accumulates and then decays; C accumulates. Each reaction converts one molecule into one, so the concentration sum is conserved. Chemical reaction changes species; binding transfers species between mobile and stationary phases.
```

## Exercise 2 (optional): reverse reactions

Keep the forward rates; set reverse rates to 0.04 and 0.03 s⁻¹ and simulate for 300 s. Predict $B/A$ and $C/B$ at equilibrium, then check the final concentrations.

In [ ]:
process = copy.deepcopy(checkpoint)
process.cycle_time = 300
reactions = MassActionLaw(process.component_system)
reactions.add_reaction(components=["A", "B"], coefficients=[-1, 1], k_fwd=0.08, k_bwd=0.04)
reactions.add_reaction(components=["B", "C"], coefficients=[-1, 1], k_fwd=0.06, k_bwd=0.03)
process.flow_sheet.reactor.bulk_reaction_model = reactions
solution = Cadet().simulate(process).solution.reactor.outlet
_ = solution.plot()
print("Final concentrations:", solution.solution[-1])

```{dropdown} Answer
Both equilibrium ratios are 2. With total concentration 1, the final concentrations approach `[1/7, 2/7, 4/7]`. Multiplying all rates by the same factor changes the relaxation time, not these ratios.
```